# 3D Atlas client — a tour

Every public method of `materials_atlas` on live data from the public 3D Atlas API, ending with
3D views of the atlas and of the embedding space. Needs the `examples` extra:

```
pip install "materials-atlas[examples]"   # + pandas, numpy, pymatgen, plotly, jupyter
```

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import materials_atlas
from materials_atlas import AtlasClient

atlas = AtlasClient()                     # the public API; AtlasClient("https://…") for another server
print("healthy:", atlas.health(), "| structures:", atlas.count())

healthy: True | structures: 210579


## One structure

`get_structure` returns the full card: properties grouped by domain plus the crystal
structure as a pymatgen dict.

In [2]:
si = atlas.get_structure("mp-149")
print(si.formula, si.structural_properties.sg_symbol, si.structural_properties.crystal_system)
print("band gap, eV:", si.electronic_properties.band_gap)
print("bulk modulus, GPa:", si.elastic_properties.bulk_modulus_voigt)
print("stable:", si.thermodynamic_properties.is_stable)
si.availability

Si Fd-3m Cubic
band gap, eV: 0.6105
bulk modulus, GPa: 88.916
stable: True


Availability(has_magnetic=True, has_elastic=True, has_dielectric=True, has_piezoelectric=False, has_embedding=True, has_projection=True, has_bandstructure=True, has_dos=True, has_xas=True)

In [3]:
crystal = si.to_pymatgen()                # pymatgen.core.Structure
print(crystal.composition, "| volume", round(crystal.volume, 2), "Å³")
crystal

Si2 | volume 40.33 Å³


Structure Summary
Lattice
    abc : 3.8492784033699095 3.8492794116013456 3.849278
 angles : 60.00001213094421 60.00000346645984 60.00001097545789
 volume : 40.32952684741405
      A : np.float64(3.333573) np.float64(0.0) np.float64(1.924639)
      B : np.float64(1.111191) np.float64(3.142924) np.float64(1.924639)
      C : np.float64(0.0) np.float64(0.0) np.float64(3.849278)
    pbc : True True True
PeriodicSite: Si (3.889, 2.75, 6.736) [0.875, 0.875, 0.875]
PeriodicSite: Si (0.5556, 0.3929, 0.9623) [0.125, 0.125, 0.125]

## Formula lookup and filtered search

Any spelling of a composition matches. `search` takes the API's filter parameters as keyword
arguments — run `print(materials_atlas.describe_filters())` for the full list with units.

In [4]:
quartz_like = atlas.find_by_formula("O2Si")
print(len(quartz_like), "SiO2 polymorphs")
materials_atlas.to_dataframe(quartz_like).sort_values("energy_above_hull").head()

325 SiO2 polymorphs


,id,formula_reduced,crystal_system,sg_number,n_atoms,volume,density,band_gap,energy_above_hull,volume_per_atom,bulk_modulus_voigt,shear_modulus_voigt,debye_temperature,thermal_conductivity_clarke,thermal_conductivity_cahill,piezoelectric_tensor_max_value,e_total,refractive_index
294,mp-7000,SiO2,Trigonal,152,9,113.625451,2.634242,5.7190,0.000000,12.625050,30.156,48.697,588.202085,1.296558,1.457014,0.193758,4.586132,1.556497
295,mp-6930,SiO2,Trigonal,154,9,113.711130,2.632258,5.6695,0.000626,12.634570,33.607,48.372,587.753591,1.317665,1.462999,0.248834,4.632816,1.566665
89,mp-12787,SiO2,Monoclinic,15,18,226.862135,2.638757,5.6273,0.002772,12.603452,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
322,mp-7029,SiO2,Tetragonal,96,12,170.841760,2.336017,5.5081,0.005306,14.236813,NaN,NaN,NaN,NaN,NaN,0.002570,4.005740,1.492060
321,mp-6945,SiO2,Tetragonal,92,12,172.838698,2.309028,5.5037,0.005371,14.403225,12.567,51.017,589.077457,1.042517,1.363464,0.001205,4.005449,1.491947


In [5]:
page = atlas.search(
    band_gap_min=1, band_gap_max=3,
    crystal_system=["Cubic", "Hexagonal"],
    is_stable=True, has_elastic=True,
    sort="bulk_modulus_voigt", order="desc",
    limit=10,
)
print(page.total, "matches, showing", len(page))
page.to_dataframe()[["id", "formula_reduced", "crystal_system", "band_gap", "bulk_modulus_voigt", "density"]]

195 matches, showing 10


,id,formula_reduced,crystal_system,band_gap,bulk_modulus_voigt,density
0,mp-1569,Be2C,Cubic,1.1637,200.504,2.466757
1,mp-3614,KTaO3,Cubic,2.0995,188.716,6.981426
2,mp-3098,AlCuO2,Hexagonal,1.8148,184.312,5.146334
3,mp-27608,Be4TeO7,Cubic,1.2807,182.130,4.206381
4,mp-730,P2Pt,Cubic,1.0186,175.121,9.109870
5,mp-804,GaN,Hexagonal,1.7265,172.144,6.080977
6,mp-20194,CeO2,Cubic,1.8647,170.992,6.994759
7,mp-5794,Zn(GaO2)2,Cubic,2.3105,168.278,6.151539
8,mp-3370,Y2Sn2O7,Cubic,2.7501,164.271,6.219572
9,mp-3924,LiNbO2,Hexagonal,1.5795,163.815,5.667092


In [6]:
# The same filter block feeds count / ids / stats
filters = dict(is_stable=True, has_dielectric=True)
print("count:", atlas.count(**filters))
stats = atlas.stats(bins=40, **filters)
hist = stats.refractive_index
px.bar(
    x=hist.bin_edges[:-1], y=hist.counts,
    labels={"x": "refractive index", "y": "structures"},
    title=f"Refractive index of {stats.total} stable structures with dielectric data",
)

count: 4517


In [7]:
# Walk every match page by page — here just to count them
n = sum(1 for _ in atlas.iter_search(band_gap_min=6, page_size=500))
print(n, "structures with band gap ≥ 6 eV")

1112 structures with band gap ≥ 6 eV


## Neighbours in embedding space

`neighbors` runs an approximate search over the MACE (default) or PET-MAD embeddings;
`similarity` is cosine similarity in percent.

In [8]:
neighbours = atlas.neighbors("mp-149", k=10)
materials_atlas.to_dataframe(neighbours)

,id,similarity,formula_reduced
0,mp-165,99.9983,Si
1,mp-1079297,99.9000,Si
2,mp-999200,99.8670,Si
3,mp-1196961,99.8465,Si
4,mp-971662,99.8464,Si
5,mp-1203790,99.8308,Si
6,mp-1204627,99.8271,Si
7,mp-1220969,99.8262,NaSi68
8,mp-1220929,99.8253,NaSi34
9,mp-1199894,99.7958,Si


## The atlas in 3D

`projection()` returns the whole point cloud (~155k points, one call). The plot below draws a
random 30k-point sample of it in grey and highlights three groups on top:

* structures with a band gap of at least 5 eV, selected with `ids(...)`;
* the SiO2 polymorphs from `find_by_formula`, located with `get_3d_coords`;
* the 50 nearest neighbours of silicon.

In [9]:
cloud = atlas.projection()
df_cloud = cloud.to_dataframe()
print(cloud.count, "points; kinds:", df_cloud["kind"].value_counts().to_dict())

wide_gap = set(atlas.ids(band_gap_min=5))
sio2 = atlas.get_3d_coords(quartz_like)                       # > 50 ids → one /projection call
si_neighbours = atlas.get_3d_coords(atlas.neighbors("mp-149", k=50))
print("SiO2 points in the atlas:", sio2.count, "| without a point:", len(sio2.missing))

155311 points; kinds: {'other': 154284, 'original': 1027}


SiO2 points in the atlas: 321 | without a point: 4


In [10]:
sample = df_cloud.sample(30_000, random_state=0)

fig = go.Figure()
fig.add_scatter3d(
    x=sample.x, y=sample.y, z=sample.z, mode="markers", name="atlas (30k sample)",
    marker=dict(size=1.5, color="lightgrey", opacity=0.4), hoverinfo="skip",
)
gap = df_cloud[df_cloud.id.isin(wide_gap)]
fig.add_scatter3d(
    x=gap.x, y=gap.y, z=gap.z, mode="markers", name="band gap ≥ 5 eV",
    marker=dict(size=2, color="royalblue", opacity=0.6), text=gap.id, hoverinfo="text",
)
fig.add_scatter3d(
    x=sio2.x, y=sio2.y, z=sio2.z, mode="markers", name="SiO2 polymorphs",
    marker=dict(size=4, color="orange"), text=sio2.ids, hoverinfo="text",
)
fig.add_scatter3d(
    x=si_neighbours.x, y=si_neighbours.y, z=si_neighbours.z, mode="markers", name="Si neighbours",
    marker=dict(size=5, color="crimson", symbol="diamond"), text=si_neighbours.ids, hoverinfo="text",
)
fig.update_layout(
    title="3D Atlas point cloud", height=700, margin=dict(l=0, r=0, t=40, b=0),
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    legend=dict(itemsizing="constant"),
)
fig

## Embeddings

`get_embeddings` fetches the vectors of a set of structures — ids, records from another call,
anything with an `.id`. Below: MACE embeddings of five compositions, projected to 3D with PCA.

In [11]:
formulas = ["Si", "SiO2", "NaCl", "Fe2O3", "TiO2"]
groups = {f: atlas.find_by_formula(f) for f in formulas}
records = [r for group in groups.values() for r in group]
emb = atlas.get_embeddings(records)                           # model="mace"
print(len(emb), "vectors of dimension", emb.dimension, "| missing:", len(emb.missing))

X = emb.to_numpy()
X_centered = X - X.mean(axis=0)
_, _, vt = np.linalg.svd(X_centered, full_matrices=False)
with np.errstate(all="ignore"):          # spurious warning from Accelerate BLAS on macOS
    pcs = X_centered @ vt[:3].T
labels = {r.id: f for f, group in groups.items() for r in group}
df_pca = pd.DataFrame(pcs, columns=["PC1", "PC2", "PC3"]).assign(
    id=emb.ids, formula=[labels[i] for i in emb.ids]
)
px.scatter_3d(
    df_pca, x="PC1", y="PC2", z="PC3", color="formula", hover_name="id",
    title="MACE embeddings, first three principal components", height=650,
).update_traces(marker_size=4)

446 vectors of dimension 256 | missing: 0


In [12]:
# Cosine similarity computed from the vectors reproduces what /neighbors reports
si_neighbours10 = atlas.neighbors("mp-149", k=10)
vectors = atlas.get_embeddings(["mp-149", *si_neighbours10])
unit = vectors.to_numpy()
unit = unit / np.linalg.norm(unit, axis=1, keepdims=True)
with np.errstate(all="ignore"):          # spurious warning from Accelerate BLAS on macOS
    cosine = unit[1:] @ unit[0]
pd.DataFrame({
    "id": [n.id for n in si_neighbours10],
    "formula": [n.formula_reduced for n in si_neighbours10],
    "similarity (server)": [n.similarity for n in si_neighbours10],
    "similarity (vectors)": np.round(cosine * 100, 4),
})

,id,formula,similarity (server),similarity (vectors)
0,mp-165,Si,99.9983,99.9983
1,mp-1079297,Si,99.9000,99.9000
2,mp-999200,Si,99.8670,99.8670
3,mp-1196961,Si,99.8465,99.8465
4,mp-971662,Si,99.8464,99.8464
5,mp-1203790,Si,99.8308,99.8308
6,mp-1204627,Si,99.8271,99.8271
7,mp-1220969,NaSi68,99.8262,99.8262
8,mp-1220929,NaSi34,99.8253,99.8253
9,mp-1199894,Si,99.7958,99.7958


In [13]:
# PET-MAD vectors are missing for ~5% of structures — they land in `missing`, nothing raises
petmad = atlas.get_embeddings(atlas.search(is_stable=True, limit=40), model="petmad")
print(len(petmad), "vectors of", petmad.dimension, "dims; missing:", petmad.missing)

39 vectors of 512 dims; missing: ['mp-10056']


## Search by a vector you computed

`neighbors` also takes an embedding vector instead of an id and runs the same search through
`POST /structures/neighbors` — for embeddings produced outside the atlas by the same MACE or
PET-MAD model. The model is inferred from the length (256 → `mace`, 512 → `petmad`). Nothing
is excluded from the result, so a structure's own vector returns that structure first.

Halfway between silicon and germanium in embedding space sit the SiGe polymorphs:

In [14]:
si_vec, ge_vec = atlas.get_embeddings(["mp-149", "mp-32"]).to_numpy()   # Si, Ge
print("own vector →", [(n.id, n.similarity) for n in atlas.neighbors(si_vec, k=2)])
materials_atlas.to_dataframe(atlas.neighbors((si_vec + ge_vec) / 2, k=8))

own vector → [('mp-149', 100.0), ('mp-165', 99.9983)]


,id,similarity,formula_reduced
0,mp-1096549,99.9196,SiGe
1,mp-978534,99.8742,SiGe
2,mp-1219182,99.8684,SiGe
3,mp-1094056,96.1792,Si7Ge
4,mp-1221235,94.2012,Na3Si34
5,mp-1199894,94.1880,Si
6,mp-1220929,94.1745,NaSi34
7,mp-1203790,94.1600,Si


In [15]:
si_vec, ge_vec = atlas.get_embeddings(["mp-149", "mp-32"], model="petmad").to_numpy()   # Si, Ge
print("own vector →", [(n.id, n.similarity) for n in atlas.neighbors(si_vec, k=2, model="petmad")])
materials_atlas.to_dataframe(atlas.neighbors((si_vec + ge_vec) / 2, k=8, model="petmad"))

own vector → [('mp-149', 100.0), ('mp-1094056', 99.8476)]


,id,similarity,formula_reduced
0,mp-1096549,99.8551,SiGe
1,mp-1219182,99.7838,SiGe
2,mp-978534,99.2126,SiGe
3,mp-1094056,98.7916,Si7Ge
4,mp-149,97.8763,Si
5,mp-1091415,97.7798,Ge
6,mp-32,97.7445,Ge
7,mp-1007760,97.5017,Ge


In [16]:
# neighbors_batch: one request per vector, an EmbeddingSet or a 2-D array; results in input order.
# For the five-composition set from above, the closest hit of every vector is the structure itself.
hits = atlas.neighbors_batch(emb, k=3)
pd.DataFrame({
    "id": emb.ids,
    "formula": [labels[i] for i in emb.ids],
    "top hit": [h[0].id for h in hits],
    "2nd": [f"{h[1].formula_reduced} {h[1].id} ({h[1].similarity:.2f})" for h in hits],
}).head(8)

,id,formula,top hit,2nd
0,mp-1245041,Si,mp-1245041,Si mp-1244971 (99.99)
1,mp-1072544,Si,mp-1072544,Si mp-1204046 (99.65)
2,mp-1544031,Si,mp-1544031,Si mp-1095269 (99.84)
3,mp-1200830,Si,mp-1200830,Si mp-1202745 (99.99)
4,mp-1201492,Si,mp-1201492,Si mp-1200830 (99.99)
5,mp-1202745,Si,mp-1202745,Si mp-1200830 (99.99)
6,mp-2416325,Si,mp-2416325,Si mp-1245041 (99.86)
7,mp-644693,Si,mp-644693,Si mp-1120447 (99.69)


## Async

`AsyncAtlasClient` mirrors every method; notebooks allow top-level `await`. Batch methods run
card requests concurrently, which matters when fetching embeddings for hundreds of ids.

In [17]:
from materials_atlas import AsyncAtlasClient

async with AsyncAtlasClient() as aatlas:
    emb_async = await aatlas.get_embeddings(quartz_like)          # 8 card requests in flight
    widest = await aatlas.search(sort="band_gap", order="desc", limit=5)
print(len(emb_async), "SiO2 vectors; widest gaps:", [b.formula_reduced for b in widest])

325 SiO2 vectors; widest gaps: ['He', 'He', 'He', 'He', 'Ne']


## Errors

Everything the client raises is an `AtlasError`; API errors carry the status code and the
server's message. Bad filter names are rejected before any request is made.

In [18]:
from materials_atlas import AtlasError, NotFoundError

for call in (
    lambda: atlas.get_structure("mp-does-not-exist"),
    lambda: atlas.search(bandgap_min=1),
    lambda: atlas.search(band_gap_min=3, band_gap_max=1),
):
    try:
        call()
    except (AtlasError, TypeError, ValueError) as error:
        print(f"{type(error).__name__}: {error}"[:120])

NotFoundError: HTTP 404: Structure mp-does-not-exist not found
TypeError: unknown filter(s): bandgap_min. Valid filters: band_gap_max, band_gap_min, bulk_modulus_voigt_max, bulk_modul
ValueError: band_gap_min (3) must not exceed band_gap_max (1)


In [19]:
atlas.close()